In [1]:
#import modules and data
import numpy as np
import gsw
import pandas as pd
import xarray as xr

gebco = xr.open_dataset('GEBCO_2025_sub_ice.nc')
geomar = xr.open_dataset('MSS_all_cruises_TEMP_PSAL.nc')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# =====================================================
# 0) Device setup (Apple Silicon GPU via MPS)
# =====================================================
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# 1) Load and clean data
# =====================================================
df = pd.read_csv("data/all_geomar_data.csv")

# Force numeric & drop invalid rows
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna().reset_index(drop=True)

# =====================================================
# 2) Define inputs and target
# =====================================================
feature_cols = [
    "N2", "LN2",
    "S", "T", "Z",
    "dSdz", "dTdz",
    "hab", "lat"
]

target_col = "EPS"

# Safety check
missing = set(feature_cols + [target_col]) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[feature_cols].values
y = df[target_col].values.reshape(-1, 1)

# =====================================================
# 3) Scale inputs
# =====================================================
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

print(f"Dataset size: {X_tensor.shape[0]} samples")

# =====================================================
# 4) Neural network
# =====================================================
class Net(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# =====================================================
# 5) 10-fold Cross-Validation
# =====================================================
kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_losses = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_tensor), 1):
    print(f"\n=== Fold {fold} ===")

    X_train = X_tensor[train_idx]
    y_train = y_tensor[train_idx]
    X_test  = X_tensor[test_idx]
    y_test  = y_tensor[test_idx]

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=1024,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=1024,
        shuffle=False
    )

    model = Net(in_dim=X_tensor.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    # ---- Training ----
    for epoch in range(30):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    # ---- Evaluation ----
    model.eval()
    with torch.no_grad():
        preds = []
        targets = []
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds.append(model(xb))
            targets.append(yb)

        preds = torch.cat(preds)
        targets = torch.cat(targets)
        fold_loss = criterion(preds, targets).item()

    fold_losses.append(fold_loss)
    print(f"Fold {fold} MSE: {fold_loss:.4e}")

print("\nMean CV MSE:", np.mean(fold_losses))

# =====================================================
# 6) Train final model on ALL data
# =====================================================
final_model = Net(in_dim=X_tensor.shape[1]).to(device)
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

full_loader = DataLoader(
    TensorDataset(X_tensor, y_tensor),
    batch_size=1024,
    shuffle=True
)

print("\nTraining final model...")
for epoch in range(20):
    final_model.train()
    for xb, yb in full_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        loss = criterion(final_model(xb), yb)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | MSE = {loss.item():.4e}")

# =====================================================
# 7) Predict EPS
# =====================================================
final_model.eval()
with torch.no_grad():
    eps_pred = final_model(X_tensor.to(device)).cpu().numpy().flatten()

# =====================================================
# 8) Post-process derived variables
# =====================================================
output_df = df.copy()
output_df["EPS_pred"] = eps_pred

# Numerical safety
eps_safe = np.clip(eps_pred, 1e-12, None)
N2_safe  = np.clip(output_df["N2"].values, 1e-12, None)

output_df["Leps_pred"] = np.log(eps_safe)
output_df["K_pred"]    = 0.2 * eps_safe / N2_safe
output_df["LK_pred"]   = np.log(np.clip(output_df["K_pred"], 1e-12, None))

# =====================================================
# 9) Save output
# =====================================================
output_df.to_csv("dataset_with_predicted_eps.csv", index=False)

print("\n✅ Saved: dataset_with_predicted_eps.csv")
print(output_df[["EPS", "EPS_pred", "Leps_pred", "K_pred", "LK_pred"]].head())
